# MLflow RAGAS Scores Query

This notebook fetches `ragas_scores.json` artifacts from MLflow runs and computes mean scores grouped by:
- `question_class`
- `subdomain`
- (`question_class`, `subdomain`)

In [ ]:
import json
import os
from pathlib import Path

import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# Config
MLFLOW_TRACKING_URI = "http://localhost:8567"
EXPERIMENT_NAME =  "exp_qsd" #None  # Set to a name string to filter one experiment
MAX_RUNS = 200

# Use one of: "latest", "all", or a specific run_id
RUN_SELECTION = "latest" #"latest"

RAGAS_TABLE_ARTIFACT = "ragas_scores.json"
METRIC_COLS = [
    "faithfulness",
    "context_precision",
    "context_recall",
    "answer_relevance",
    "factual_correctness",
    "factual_correctness_recall"
]

# question_class values to exclude from subdomain and overall aggregations.
# "unanswerable" questions score 0 on retrieval/relevance metrics by design,
# so including them in subdomain averages would unfairly dilute results.
EXCLUDE_QUESTION_CLASSES = ["unanswerable"]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
def list_artifacts_recursive(run_id: str, path: str = ""):
    items = []
    for art in client.list_artifacts(run_id, path):
        items.append(art.path)
        if art.is_dir:
            items.extend(list_artifacts_recursive(run_id, art.path))
    return items


def normalize_logged_table(json_obj) -> pd.DataFrame:
    # mlflow.log_table can be stored in different JSON shapes depending on version.
    if isinstance(json_obj, list):
        return pd.DataFrame(json_obj)

    if isinstance(json_obj, dict):
        if "data" in json_obj and "columns" in json_obj:
            return pd.DataFrame(json_obj["data"], columns=json_obj["columns"])
        return pd.DataFrame(json_obj)

    raise ValueError("Unsupported ragas_scores.json structure")


def load_ragas_table_for_run(run_id: str):
    artifact_paths = list_artifacts_recursive(run_id)

    # Find ragas_scores.json at any artifact depth.
    matches = [p for p in artifact_paths if p.endswith(RAGAS_TABLE_ARTIFACT)]
    if not matches:
        return None, None

    artifact_path = matches[0]
    local_path = client.download_artifacts(run_id, artifact_path)

    with open(local_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    df = normalize_logged_table(payload)
    return df, artifact_path

In [ ]:
# Discover runs
if EXPERIMENT_NAME:
    exp = client.get_experiment_by_name(EXPERIMENT_NAME)
    if exp is None:
        raise ValueError(f"Experiment not found: {EXPERIMENT_NAME}")
    experiment_ids = [exp.experiment_id]
else:
    experiment_ids = [e.experiment_id for e in client.search_experiments()]

runs_df = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["attribute.start_time DESC"],
    max_results=MAX_RUNS,
)

if runs_df.empty:
    raise ValueError("No MLflow runs found for the current filter.")

print(f"Found {len(runs_df)} runs")
runs_df[["run_id", "experiment_id", "start_time", "status"]].head(10)

In [ ]:
# Pull ragas_scores.json from each run
tables = []
misses = []

for run_id in runs_df["run_id"].tolist():
    df, artifact_path = load_ragas_table_for_run(run_id)
    if df is None:
        misses.append(run_id)
        continue

    df = df.copy()
    df["run_id"] = run_id
    df["artifact_path"] = artifact_path
    tables.append(df)

if not tables:
    raise ValueError("No ragas_scores.json artifacts found in discovered runs.")

all_scores = pd.concat(tables, ignore_index=True)

for col in METRIC_COLS:
    if col in all_scores.columns:
        all_scores[col] = pd.to_numeric(all_scores[col], errors="coerce")

print(f"Runs with ragas_scores.json: {all_scores['run_id'].nunique()}")
print(f"Runs without ragas_scores.json: {len(misses)}")
all_scores.head()

In [ ]:
# Select target rows for aggregation
if RUN_SELECTION == "latest":
    latest_run_id = runs_df.iloc[0]["run_id"]
    target = all_scores[all_scores["run_id"] == latest_run_id].copy()
    print(f"Using latest run: {latest_run_id}")
elif RUN_SELECTION == "all":
    target = all_scores.copy()
    print("Using all runs with ragas_scores.json")
else:
    target = all_scores[all_scores["run_id"] == RUN_SELECTION].copy()
    if target.empty:
        raise ValueError(f"No rows found for run_id={RUN_SELECTION}")
    print(f"Using selected run: {RUN_SELECTION}")

required_cols = ["question_class", "subdomain"]
for c in required_cols:
    if c not in target.columns:
        raise ValueError(f"Missing required column in ragas table: {c}")

target[["run_id", "user_input", "question_class", "subdomain"] + [c for c in METRIC_COLS if c in target.columns]].head()

In [ ]:
metric_cols_present = [c for c in METRIC_COLS if c in target.columns]

# Rows used for subdomain aggregations — excludes question classes whose
# zero scores are expected (e.g. unanswerable) and would dilute averages.
target_for_subdomain = (
    target[~target["question_class"].isin(EXCLUDE_QUESTION_CLASSES)].copy()
    if EXCLUDE_QUESTION_CLASSES
    else target
)
if EXCLUDE_QUESTION_CLASSES:
    print(f"Excluding from subdomain aggregation: {EXCLUDE_QUESTION_CLASSES}")

avg_by_question_class = (
    target.groupby("question_class", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_subdomain = (
    target_for_subdomain.groupby("subdomain", dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

avg_by_qclass_and_subdomain = (
    target.groupby(["question_class", "subdomain"], dropna=False)[metric_cols_present]
    .mean(numeric_only=True)
    .sort_index()
)

print("Average by question_class")
display(avg_by_question_class)

print(f"Average by subdomain (excluding: {EXCLUDE_QUESTION_CLASSES or 'none'})")
display(avg_by_subdomain)

print("Average by (question_class, subdomain)")
display(avg_by_qclass_and_subdomain)